# ERIKA, Part 2: Agent for existing records

This notebook opens an **already existing** ERIKA record folder and builds a LangChain agent over it. It does not call the data generator, create patient records, or write teaching cases. Run Part 1 first, or point `DATA_DIR` to an existing folder with the same schema.

**Expected folder:** `manifest.json` (synthetic fixture version 4), `appointments.json`, `patients/*.json`, `documents/`, and `fictional_training_cases.json`. The current tools expect this schema. Other existing EHR exports or loose PDFs need an import/mapping step before these tools can query them.

**Run order:** Configure Bedrock, set `DATA_DIR`, load and validate records, define teaching functions, build the agent, then ask questions. Bedrock calls may incur charges. Keep the token in your environment or `.env`.

## 1. Install dependencies and configure Bedrock

In [ ]:
# Install required packages (run once)
import subprocess, sys

print('📦 Installing required packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'langchain>=0.3.25',
    'langchain-aws>=0.2.24',
    'langchain-community>=0.3.23',
    'langchain-core>=0.3.62',
    'boto3>=1.38.0',
    'pydantic>=2.11.4',
    'python-dotenv>=1.0',
], check=True)
print('✅ All packages installed!')

In [ ]:
import os, json, math
from typing import Any, Optional
from datetime import datetime

# Load API key from .env file (never hardcode secrets in notebooks!)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('✅ python-dotenv loaded')
except ImportError:
    print('⚠️  python-dotenv not installed — run: pip install python-dotenv')

from langchain_core.tools import Tool, tool
print('✅ langchain_core.tools imported')

from langchain_core.prompts import PromptTemplate
print('✅ langchain_core.prompts imported')

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
print('✅ langchain_aws imported')

# LangChain 1.x uses create_agent (replaces create_react_agent + AgentExecutor)
from langchain.agents import create_agent
print('✅ create_agent imported (LangChain 1.x API)')

# ── AWS Configuration ──────────────────────────────────────
# Key is read from .env file (AWS_BEARER_TOKEN_BEDROCK=your_key_here)
BEDROCK_API_KEY = os.getenv('AWS_BEARER_TOKEN_BEDROCK', '')
AWS_REGION = 'ap-southeast-2'
MODEL_ID   = 'global.amazon.nova-2-lite-v1:0'

if not BEDROCK_API_KEY:
    BEDROCK_API_KEY = input('⚠️  AWS_BEARER_TOKEN_BEDROCK not found in .env\nEnter your Bearer Token: ').strip()

os.environ['AWS_BEARER_TOKEN_BEDROCK'] = BEDROCK_API_KEY
os.environ['AWS_DEFAULT_REGION']       = AWS_REGION

print(f'\n✅ AWS environment configured')
print(f'   Region : {AWS_REGION}')
print(f'   Model  : {MODEL_ID}')
print('\n' + '='*70)
print('🚀 READY TO BUILD AGENTS! (LangChain 1.x)')
print('='*70)

## 2. Check the model connection

In [ ]:
# STEP 1: Initialize LLM with AWS Bedrock

print('🤖 Initializing AWS Bedrock LLM...\n')

llm = ChatBedrockConverse(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
    max_tokens=512,
)

test = llm.invoke([HumanMessage(content='Say: AWS Bedrock agent ready!')])
print('✅ LLM initialized successfully!')
print(f'   Model  : {MODEL_ID}')
print(f'   Region : {AWS_REGION}')
print(f'   Test   : {test.content[:60]}...')

## 3. Define the existing-record reader

This cell contains the store and its methods. It deliberately omits `build_fixture` and `generate_synthetic_files`. The payment and ingestion methods remain available as explicit tools; merely opening the store does not write files.

In [ ]:
"""Open-access file tools for generated fictional hospital records only."""
from __future__ import annotations
import json
import uuid
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path


def _save_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)


class OpenSyntheticStore:
    """No roles or patient isolation. Every fictional record is queryable."""
    def __init__(self, folder: str | Path):
        self.root = Path(folder).expanduser().resolve()
        self.manifest = json.loads((self.root / "manifest.json").read_text(encoding="utf-8"))
        if not (self.manifest.get("synthetic") is True and self.manifest.get("version") == 4):
            raise ValueError("Expected version 4 fictional files")
        self.patient_ids = tuple(self.manifest["patient_ids"])

    def _record(self, patient_id: str) -> dict | None:
        if patient_id not in self.patient_ids:
            return None
        return json.loads((self.root / "patients" / f"{patient_id}.json").read_text(encoding="utf-8"))

    def _records(self):
        for pid in self.patient_ids:
            yield self._record(pid)

    def hospital_stats(self) -> dict:
        encounters = [e for record in self._records() for e in record["encounters"]]
        inpatient = [e for e in encounters if e["kind"] == "inpatient"]
        return {"hospital_visits": len(encounters), "unique_patients": len(self.patient_ids),
                "admissions": len(inpatient),
                "discharges": sum(e["discharge_date"] is not None for e in inpatient),
                "currently_admitted": sum(e["discharge_date"] is None for e in inpatient)}

    def list_patient_names(self, group: str = "all") -> list[dict]:
        if group not in {"all", "admitted", "discharged", "currently_admitted"}:
            raise ValueError("group must be all/admitted/discharged/currently_admitted")
        result = []
        for r in self._records():
            admissions = [e for e in r["encounters"] if e["kind"] == "inpatient"]
            if group == "admitted" and not admissions: continue
            if group == "discharged" and not any(e["discharge_date"] for e in admissions): continue
            if group == "currently_admitted" and not any(e["discharge_date"] is None for e in admissions): continue
            result.append({k: r["patient"][k] for k in ("patient_id", "name")})
        return result

    def patient_record(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        return {k: r[k] for k in ("patient", "encounters", "documents")} if r else {"error": "Patient not found"}

    def list_documents(self, patient_id: str = "") -> list[dict]:
        return [{k: d[k] for k in ("document_id", "patient_id", "encounter_id", "file_name", "source_type")}
                for r in self._records() if not patient_id or r["patient"]["patient_id"] == patient_id
                for d in r["documents"]]

    def _text(self, document: dict) -> str:
        path = (self.root / document["path"]).resolve()
        if not path.is_relative_to(self.root / "documents"):
            raise ValueError("Invalid document metadata path")
        return path.read_text(encoding="utf-8")

    def read_document(self, document_id: str) -> dict:
        for r in self._records():
            for d in r["documents"]:
                if d["document_id"] == document_id:
                    return {k: d[k] for k in ("document_id", "patient_id", "encounter_id", "file_name", "source_type")} | {"text": self._text(d)}
        return {"error": "Document not found"}

    def search_documents(self, query: str, patient_id: str = "") -> list[dict]:
        if not query.strip(): raise ValueError("Enter a search phrase")
        matches = []
        for d in self.list_documents(patient_id):
            text = self.read_document(d["document_id"])["text"]
            if query.casefold() in text.casefold():
                matches.append(d | {"preview": text[:250]})
        return matches

    def available_appointments(self, doctor: str = "", date: str = "") -> list[dict]:
        slots = json.loads((self.root / "appointments.json").read_text(encoding="utf-8"))["slots"]
        return [{k: slot[k] for k in ("slot_id", "doctor", "date", "time")}
                for slot in slots if slot["status"] == "available"
                and (not doctor or slot["doctor"] == doctor)
                and (not date or slot["date"] == date)]

    def calculate_bmi(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        h, w = r["patient"]["height_cm"], r["patient"]["weight_kg"]
        return {"patient_id": patient_id, "height_cm": h, "weight_kg": w,
                "bmi": round(w / (h / 100) ** 2, 1), "formula": "weight_kg / (height_cm / 100)^2"}

    def billing_balance(self, patient_id: str) -> dict:
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        total = r["invoice"]["total_cents"]
        paid = sum(p["amount_cents"] for p in r["payments"])
        return {"patient_id": patient_id, "currency": "CAD", "invoice_id": r["invoice"]["invoice_id"],
                "total_cents": total, "paid_cents": paid, "balance_cents": max(0, total - paid),
                "credit_cents": max(0, paid - total), "payments": r["payments"]}

    def post_synthetic_payment(self, patient_id: str, amount_cad: str, receipt_id: str) -> dict:
        """Change fictional ledger only; repeat receipt IDs are idempotent."""
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        try: value = Decimal(amount_cad)
        except InvalidOperation as exc: raise ValueError("Invalid amount") from exc
        cents = value * 100
        if not value.is_finite() or cents <= 0 or cents != cents.to_integral_value():
            raise ValueError("Use a positive CAD amount with up to two decimals")
        if not receipt_id or len(receipt_id) > 80: raise ValueError("Receipt ID required")
        for other in self._records():
            for payment in other["payments"]:
                if payment["payment_id"] == receipt_id:
                    if payment["patient_id"] != patient_id or payment["amount_cents"] != int(cents):
                        raise ValueError("Receipt already used for different details")
                    return self.billing_balance(patient_id)
        r["payments"].append({"payment_id": receipt_id, "patient_id": patient_id,
            "amount_cents": int(cents), "recorded_date": datetime.now(timezone.utc).date().isoformat(),
            "source": "synthetic model test payment"})
        _save_json(self.root / "patients" / f"{patient_id}.json", r)
        return self.billing_balance(patient_id)

    def ingest_local_document(self, patient_id: str, local_path: str | Path) -> dict:
        """Notebook-side upload simulation; .txt/.md/.pdf/.docx, not a model tool."""
        r = self._record(patient_id)
        if not r: return {"error": "Patient not found"}
        source = Path(local_path)
        if not source.is_file() or source.stat().st_size > 2_000_000:
            raise ValueError("File missing or larger than 2 MB")
        if source.suffix.lower() in {".txt", ".md"}:
            text = source.read_text(encoding="utf-8")
        elif source.suffix.lower() == ".pdf":
            from pypdf import PdfReader
            text = "\n".join(page.extract_text() or "" for page in PdfReader(source).pages[:50])
        elif source.suffix.lower() == ".docx":
            from docx import Document
            text = "\n".join(p.text for p in Document(source).paragraphs)
        else: raise ValueError("Supported types: .txt, .md, .pdf, .docx")
        if not text.strip() or len(text) > 200_000:
            raise ValueError("No extractable text or too long")
        doc_id = "UPLOAD-" + uuid.uuid4().hex[:12]
        relative = f"documents/{patient_id}/{doc_id}.txt"
        dest = self.root / relative
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(text, encoding="utf-8")
        r["documents"].append({"document_id": doc_id, "patient_id": patient_id,
            "encounter_id": None, "file_name": source.name, "source_type": "local_upload", "path": relative})
        _save_json(self.root / "patients" / f"{patient_id}.json", r)
        return {"patient_id": patient_id, "document_id": doc_id, "characters": len(text)}

## 4. Select and validate the records folder

By default this looks for the folder alongside the notebook's current working directory. Set `DATA_DIR` to an absolute path if the records are elsewhere. The cell raises a useful error if files are missing; it never generates replacements.

In [ ]:
from pathlib import Path

DATA_DIR = Path.cwd() / "hospital_open_synthetic_data"
# Example: DATA_DIR = Path(r"C:\Users\YourName\Documents\hospital_open_synthetic_data")
DATA_DIR = DATA_DIR.expanduser().resolve()
required = [DATA_DIR / "manifest.json", DATA_DIR / "appointments.json",
            DATA_DIR / "fictional_training_cases.json", DATA_DIR / "patients",
            DATA_DIR / "documents"]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Existing ERIKA records were not found. Set DATA_DIR to the folder "
        "created by Part 1. Missing: " + ", ".join(missing)
    )

store = OpenSyntheticStore(DATA_DIR)
CASE_PATH = DATA_DIR / "fictional_training_cases.json"
# Fail early if a patient file or note referenced by the manifest is missing.
for patient_id in store.patient_ids:
    record = store._record(patient_id)
    for document in record["documents"]:
        note_path = (DATA_DIR / document["path"]).resolve()
        if not note_path.is_relative_to(DATA_DIR) or not note_path.is_file():
            raise FileNotFoundError(f"Missing or invalid note for {patient_id}: {document['path']}")
print("Records loaded from:", DATA_DIR)
print("Patient records:", len(store.patient_ids))
print("Available notes:", len(store.list_documents()))
print("Visits:", store.hospital_stats()["hospital_visits"])

## 5. Define clinical teaching functions

These functions read the existing records and teaching-case file. They do not seed either file.

In [ ]:
"""Open synthetic educational scaffolding for fictional records.

Source links are official publisher/guideline pages. Summaries are teaching
fixtures, not locally clinician-approved content or current treatment advice.
"""
from __future__ import annotations
import json
from pathlib import Path
HospitalStore = OpenSyntheticStore

SOURCE_CATALOG = {
    "hypertension": {
        "overview": "A documented high blood pressure condition; interpretation depends on properly obtained measurements and clinical context.",
        "publisher": "Hypertension Canada", "title": "Guidelines",
        "url": "https://hypertension.ca/guidelines/",
    },
    "type 2 diabetes": {
        "overview": "A documented condition involving blood glucose regulation; review the source record and diagnostic context.",
        "publisher": "Diabetes Canada", "title": "Definition, Classification and Diagnosis of Diabetes",
        "url": "https://guidelines.diabetes.ca/cpg/chapter3",
    },
    "asthma": {
        "overview": "A condition involving variable respiratory symptoms and airway inflammation; assess the actual case rather than assuming a symptom establishes asthma.",
        "publisher": "Global Initiative for Asthma", "title": "GINA reports",
        "url": "https://ginasthma.org/reports/",
    },
    "chronic kidney disease": {
        "overview": "A kidney structure or function condition whose chronicity and classification require the appropriate history and investigations.",
        "publisher": "KDIGO", "title": "2024 CKD Evaluation and Management Guideline",
        "url": "https://kdigo.org/guidelines/ckd-evaluation-and-management/",
    },
    "migraine": {
        "overview": "A headache disorder; a new or unusual headache requires its own clinical assessment.",
        "publisher": "NICE", "title": "Headaches in over 12s: diagnosis and management",
        "url": "https://www.nice.org.uk/guidance/cg150",
    },
    "osteoarthritis": {
        "overview": "A joint condition; symptoms, examination and function matter to the clinical assessment.",
        "publisher": "NICE", "title": "Osteoarthritis in over 16s: diagnosis and management",
        "url": "https://www.nice.org.uk/guidance/ng226",
    },
    "heart failure": {
        "overview": "A clinical syndrome involving impaired cardiac function; type, severity and treatment require clinician assessment.",
        "publisher": "Canadian Cardiovascular Society", "title": "Heart failure guidelines and tools",
        "url": "https://ccs.ca/guidelines/",
    },
    "iron deficiency anemia": {
        "overview": "A documented anemia label; confirm the source's measurements and the clinician's assessment of cause.",
        "publisher": "World Health Organization", "title": "Guideline on haemoglobin cutoffs to define anaemia",
        "url": "https://www.who.int/publications/i/item/9789240088542",
    },
}


def search_clinical_education(store: HospitalStore, topic: str) -> dict:
    key = topic.strip().lower()
    matches = [dict(topic=name, **entry) for name, entry in SOURCE_CATALOG.items()
               if key and (key in name or name in key)]
    return {"matches": matches, "review_status": "teaching fixture; local clinical approval pending",
            "note": "Open the linked primary source to verify current recommendations; this catalog contains no dosing advice."}


def explain_documented_diagnosis(store: HospitalStore, patient_id: str,
                                  encounter_id: str) -> dict:
    record = store.patient_record(patient_id)
    encounter = next((e for e in record["encounters"]
                      if e["encounter_id"] == encounter_id), None)
    if encounter is None:
        return {"error": "Encounter not found for this patient"}
    topic = encounter["documented_diagnosis"].lower()
    document_id = f"DOC-{encounter_id}"
    source = store.read_document(document_id)
    reference = SOURCE_CATALOG.get(topic)
    return {"patient_id": patient_id, "encounter_id": encounter_id,
            "document_id": document_id, "documented_diagnosis": topic,
            "document_excerpt": source["text"], "education_reference": reference,
            "instruction": "Explain the documented term, separate record facts from general education, and state uncertainties. Do not choose treatment."}


def load_training_cases(path: str | Path) -> list[dict]:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if data.get("synthetic") is not True or len(data.get("cases", [])) != 5:
        raise ValueError("Expected five explicitly synthetic training cases")
    return data["cases"]


def get_training_case(store: HospitalStore, case_id: str,
                      path: str | Path) -> dict:
    case = next((c for c in load_training_cases(path) if c["case_id"] == case_id), None)
    if case is None:
        return {"error": "Training case not found"}
    return {"case": case, "review_status": "fictional teaching case; not clinical validation data"}


def prepare_case_discussion(store: HospitalStore, case_text: str) -> dict:
    if not case_text.strip() or len(case_text) > 6000:
        raise ValueError("Provide a case between 1 and 6000 characters")
    matches = [dict(topic=name, **entry) for name, entry in SOURCE_CATALOG.items()
               if name in case_text.lower()]
    urgent_example = ("sudden severe headache" in case_text.lower()
                      and "weakness" in case_text.lower())
    return {
        "case_text": case_text,
        "matched_references": matches,
        "clinician_role": "open synthetic test session",
        "discussion_structure": ["known facts", "documented versus possible explanations",
                                 "uncertainties and missing data", "safety concerns to assess",
                                 "questions for supervising clinician"],
        "teaching_questions": ["What facts are directly documented?",
                               "Which explanations remain hypotheses?",
                               "What further history, examination, or records are needed?",
                               "Which concern requires immediate human assessment?"],
        "safety_signal": ("Sudden severe headache with new weakness in this fictional example "
                          "calls for immediate emergency assessment; do not delay for the agent."
                          if urgent_example else "Clinician must assess urgency from the complete case."),
        "limits": "No definitive diagnosis, treatment selection, dosing, or autonomous triage. Human clinician must review.",
        "review_status": "teaching scaffold, not validated clinical decision support",
    }


def suggest_diagnoses_and_treatment_options(store: HospitalStore,
                                           symptoms: str) -> dict:
    """Limited, source-linked teaching hypotheses; never a diagnosis or order."""
    narrative = symptoms.strip()
    if not narrative or len(narrative) > 6000:
        raise ValueError("Provide symptoms between 1 and 6000 characters")
    lower = narrative.lower()
    # Deliberately conservative: negation is not reliably parsed by this demo.
    red_flags = {
        "new one-sided weakness": ("one-sided weakness", "arm weakness", "new weakness"),
        "sudden severe headache": ("sudden severe headache", "worst headache"),
        "chest pain": ("chest pain", "chest pressure"),
        "severe breathing difficulty": ("severe shortness of breath", "struggling to breathe"),
    }
    found_flags = [name for name, phrases in red_flags.items()
                   if any(phrase in lower for phrase in phrases)]
    if found_flags:
        return {
            "status": "urgent_human_assessment",
            "matched_safety_signals": found_flags,
            "message": "Seek immediate emergency assessment (call 911 in Ontario). Do not wait for this agent's differential or treatment suggestions.",
            "possible_diagnoses": [], "treatment_options": [],
            "source": "https://www.heartandstroke.ca/articles/if-it-might-be-a-stroke-call-911",
            "limits": "Keyword triggers are illustrative and cannot rule out an emergency. Human clinical assessment required.",
        }

    # Examples are intentionally narrow. No probability, ranking, dose, or prescription.
    examples = [
        {"terms": ("wheeze", "wheezing"), "topic": "asthma",
         "hypothesis": "Asthma is one possibility if variable symptoms and objective assessment support it.",
         "option": "Clinician may review an inhaled controller strategy and an individualized action plan against the current GINA report.",
         "check": "Current severity, examination, objective lung-function evidence, and alternative causes."},
        {"terms": ("thirst", "frequent urination"), "topic": "type 2 diabetes",
         "hypothesis": "Diabetes is one possibility; symptoms alone do not establish type or diagnosis.",
         "option": "If confirmed, clinician may review education, lifestyle support, monitoring, and medication choices under the relevant guideline.",
         "check": "Glucose testing, timeline, medications, hydration, and alternative causes."},
        {"terms": ("ankle swelling", "leg swelling"), "topic": "heart failure",
         "hypothesis": "Heart failure is one possibility; swelling alone has multiple causes.",
         "option": "If confirmed, clinician may review guideline-directed management and the person's existing plan; no medicine or dose is selected here.",
         "check": "Breathing symptoms, vital signs, volume status, medications, kidney function, and other causes."},
    ]
    candidates = []
    for example in examples:
        if any(term in lower for term in example["terms"]):
            candidates.append({"possible_diagnosis": example["topic"],
                               "why_consider": example["hypothesis"],
                               "treatment_option_for_review": example["option"],
                               "information_needed": example["check"],
                               "source": SOURCE_CATALOG[example["topic"]]["url"]})
    return {
        "status": "clinician_teaching_review" if candidates else "insufficient_catalog_coverage",
        "symptoms_as_supplied": narrative,
        "possible_diagnoses": candidates,
        "message": ("These are non-exhaustive hypotheses and treatment topics for licensed clinician review. "
                    "Check the linked current guidance and assess the complete patient before any decision.")
                    if candidates else "No catalog-supported hypothesis for this description. Ask for clinical assessment; do not guess.",
        "limits": "Fictional prototype; not validated for clinical use, triage, diagnosis, prescribing, or exclusion of conditions.",
    }

In [ ]:
find_education = search_clinical_education
explain_diagnosis = explain_documented_diagnosis
read_training_case = get_training_case
discuss_case = prepare_case_discussion
suggest_options = suggest_diagnoses_and_treatment_options
print("Teaching cases:", len(load_training_cases(CASE_PATH)))

## 6. Build ERIKA

The agent has 17 tools. It can look up existing fictional records and documents, appointments, billing, and teaching cases. Its 450-character response limit can make complex clinical explanations incomplete. The dataset and tools are an educational demonstration; real patient data requires access controls, consent, audit logging, and clinical review.

In [ ]:
import json
from pathlib import Path
from langchain.agents import create_agent
from langchain.tools import tool


AGENT_SYSTEM_PROMPT = """
You are ERIKA (AI Resource and Healthcare Assistant), a friendly hospital support agent. 
All patients and documents in the folder are available through tools. Help users with Healthcare-related questions. Always be friendly and professional. Use tools to find accurate information. If you do not have the information, say so clearly and explain the medical terms asked by the user. You can explain prognosis, 
diagnosis, treatment and medical cases provided by user but you MUST respond in 450 characters

Keep every response to 450 characters or fewer. Prioritize the direct
answer and any urgent safety information. If the question needs more detail,
offer to explain one part at a time.
"""

EDUCATION_LIBRARY = {
    "appointment": (
        "Education fixture E1: Write down what you want to discuss and bring "
        "questions to your appointment. This is general preparation guidance."
    ),
    "follow-up": (
        "Education fixture E2: Ask the clinic who arranges follow-up and how "
        "you will receive appointment details."
    ),}


def make_open_agent(store: OpenSyntheticStore, model, case_path: str | Path):

    def safe(fn, *args, **kwargs) -> str:
        try:
            return json.dumps(fn(*args, **kwargs), ensure_ascii=False)
        except (ValueError, FileNotFoundError) as exc:
            return json.dumps({"error": str(exc)})

    @tool
    def search_education_library(query: str) -> str:
        """Search topic education, not synthetic patient records."""
        query = query.lower()
        matches = [text for keyword, text in EDUCATION_LIBRARY.items()
                   if keyword in query]
        return "\n\n".join(matches) if matches else "No reviewed education fixture found."

    @tool
    def check_understanding(topic: str, patient_explanation: str) -> str:
        """Reflect the user's explanation without grading clinical accuracy."""
        return (
            f"Topic: {topic}\nUser's words: {patient_explanation}\n"
            "Clarify against a reviewed source or identify an unanswered question."
        )
    
    @tool
    def hospital_statistics() -> str:
        """Get exact counts of visits, patients, admissions, and discharges."""
        return safe(store.hospital_stats)

    @tool
    def list_patient_names(group: str = "all") -> str:
        """List patients in the all, admitted, discharged, or currently_admitted group."""
        return safe(store.list_patient_names, group)

    @tool
    def get_patient_record(patient_id: str) -> str:
        """Read a patient's history, encounters, treatments, care team, and document IDs."""
        return safe(store.patient_record, patient_id)

    @tool
    def list_available_documents(patient_id: str = "") -> str:
        """List document IDs, optionally for one patient."""
        return safe(store.list_documents, patient_id)

    @tool
    def search_documents(query: str, patient_id: str = "") -> str:
        """Search documents by phrase, optionally for one patient."""
        return safe(store.search_documents, query, patient_id)

    @tool
    def read_document(document_id: str) -> str:
        """Read the complete text of a document by ID."""
        return safe(store.read_document, document_id)

    @tool
    def available_appointment_slots(
        doctor: str = "", date: str = ""
    ) -> str:
        """Get available appointment slots."""
        return safe(store.available_appointments, doctor, date)

    @tool
    def calculate_patient_bmi(patient_id: str) -> str:
        """Calculate BMI from recorded height and weight."""
        return safe(store.calculate_bmi, patient_id)

    @tool
    def get_billing_balance(patient_id: str) -> str:
        """Get invoice, payments, and remaining CAD balance."""
        return safe(store.billing_balance, patient_id)

    @tool
    def post_synthetic_payment(
        patient_id: str, amount_cad: str, receipt_id: str
    ) -> str:
        """Record an explicitly requested payment with a unique receipt ID."""
        return safe(
            store.post_synthetic_payment,
            patient_id,
            amount_cad,
            receipt_id,
        )

    @tool
    def search_clinical_education(topic: str) -> str:
        """Find source-linked clinical teaching summaries for a documented condition."""
        return safe(find_education, store, topic)

    @tool
    def explain_documented_diagnosis(
        patient_id: str, encounter_id: str
    ) -> str:
        """Explain a diagnosis documented in a encounter."""
        return safe(explain_diagnosis, store, patient_id, encounter_id)

    @tool
    def get_training_case(case_id: str) -> str:
        """Read a medical training case by CASE ID."""
        return safe(read_training_case, store, case_id, case_path)

    @tool
    def prepare_case_discussion(case_text: str) -> str:
        """Structure an educational case discussion with facts, hypotheses, gaps and safety issues."""
        return safe(discuss_case, store, case_text)

    @tool
    def suggest_clinical_options(symptoms: str) -> str:
        """Return limited symptom-based diagnostic possibilities and treatment topics for review; urgent signals suppress suggestions."""
        return safe(suggest_options, store, symptoms)

    # IMPORTANT: This entire list is indented INSIDE make_open_agent.
    all_tools = [
        check_understanding,
        search_education_library,
        hospital_statistics,
        list_patient_names,
        get_patient_record,
        list_available_documents,
        search_documents,
        read_document,
        available_appointment_slots,
        calculate_patient_bmi,
        get_billing_balance,
        post_synthetic_payment,
        search_clinical_education,
        explain_documented_diagnosis,
        get_training_case,
        prepare_case_discussion,
        suggest_clinical_options,
    ]

    # Your existing AWS Bedrock llm is passed in as `model`.
    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=AGENT_SYSTEM_PROMPT,
    )

    return agent, all_tools


# This part is OUTSIDE the function.
erika_agent, erika_tools = make_open_agent(
    store=store,
    model=llm,
    case_path=CASE_PATH,
)

print("✅ Agent ready")
print("Tools:", [t.name for t in erika_tools])

## 7. Ask questions about the existing records

These examples call the model independently. Run any one after Section 6. They read records from `DATA_DIR`.

### 7.1 Record and visit lookup

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How many fictional patients and hospital visits are recorded?"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "For SYN-007, show admission date, discharge status and relevant note ID."
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

### 7.2 General health questions and documented explanations

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I have cold, what is the diagnosis. Also  nasal congestion"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I need educational info for cold"
        }
    ]
})


print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "provide educational material on upper‑respiratory‑infection management"
        }
    ]
})


print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what is hypertension"
        }
    ]
})


print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain the documented diagnosis at SYN-001 encounter ENC-001-1 with a source link."
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "can you perform calculations?"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what is the BMI of a patient with 56kg and 178 cm. the age of the patient is 30 year. is this a good BMI or is that a medical issue? What are the possible prognosis?"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

### 7.3 Patient list, billing, and general information

The balance example reflects payments already stored in the folder.

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "List all patients id"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is SYN-001's verified balance after the additional test payment of $100?"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what is cancer"
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

### 7.4 Fictional pasted cases

In [ ]:
result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Fictional case: A 45-year-old has four weeks of dizziness on standing,
episodic palpitations, and loose stools. A new medication was started
two weeks ago. No vital signs or test results are available.

Explain possible causes as hypotheses, what information is missing,
and what should be discussed with a clinician. This case is not in
any patient document.
"""
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

In [ ]:
#print("Case file exists:", CASE_PATH.exists())

result = erika_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Fictional discharge-note text: HFrEF with an ejection fraction of
35%; CKD stage G3b with eGFR 38 mL/min/1.73 m²; type 2 diabetes in the
history. Repeat blood tests and cardiology follow-up were requested. Iron
studies are pending. An older list says furosemide 20 mg daily, while the
discharge list says 40 mg daily. The current dose is unknown. No new
symptoms are reported.

Explain the terms and measurements in plain language. Separate what this
pasted text states from general information and what remains unknown.
Explain why the heart and kidney conditions may both matter. Identify the
medication-list discrepancy.
"""
        }
    ]
})

print("My name is ERIKA, an AI Resource and Healthcare Assistant.", result["messages"][-1].content)

## Using another existing dataset

Set `DATA_DIR` to that dataset's folder. It must follow the documented ERIKA schema; this notebook does not automatically convert arbitrary clinical records. For new source formats, add a separate ingestion adapter that maps patient IDs, encounters, note IDs and paths, appointments, and billing fields into this schema. Restrict access by patient and obtain the necessary privacy and clinical review before any use with real patient records.